# Bi-encoder Fine-tuning (Tier 1b)

Fine-tunes the first-stage retriever (`all-MiniLM-L6-v2`) on in-domain TREC/KZ clinical trial data
using `MultipleNegativesRankingLoss`. Evaluates retrieval quality on the TREC22 judged pool.

**Architecture role:** The bi-encoder is the *first-stage retriever* — it narrows 374k corpus docs
to ~10k candidates before the cross-encoder classifier. Fine-tuning it improves domain alignment
but does **not** affect `clf` NDCG in `eval_baseline.ipynb` (eval mode collapses the cascade;
all judged docs pass through regardless of retrieval score). The right evaluation here is
bi-encoder ranking within the judged pool.

**Training data:** Positive (topic, trial) pairs from `train_clf_data.jsonl` — label=2 only.
TREC22 is excluded from training (same clean split as the classifier).

**Evaluation:** Within the TREC22 judged pool (35,394 pairs, 50 topics) — cosine ranking
of judged docs per topic. This is a proxy for real retrieval recall over 374k docs;
full-corpus evaluation would require re-embedding ~374k docs (deferred).

**Steps (run top to bottom):**
1. GPU check → Install → HF token → Mount Drive → Config
2. Load training data (positive pairs from TREC21+KZ)
3. Fine-tune MiniLM with MultipleNegativesRankingLoss
4. Save to Drive → push to Hub
5. Load TREC22 judged pool for evaluation
6. Evaluate retrieval quality: off-the-shelf MiniLM vs fine-tuned MiniLM
7. Results table + §7b numbers for deep dive

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — training will be slow')

In [ ]:
!pip install -q sentence-transformers datasets huggingface_hub

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # paste your WRITE token (huggingface.co/settings/tokens)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ===== PATHS =================================================================
DATA_ROOT      = '/content/drive/MyDrive/ct_data23'
TRAIN_FILE     = os.path.join(DATA_ROOT, 'train_clf_data.jsonl')   # TREC21+KZ, no TREC22
QRELS_FILE     = os.path.join(DATA_ROOT, 'unified_qrels.jsonl')    # all 184 topics
OUTPUT_DIR     = os.path.join(DATA_ROOT, 'models', 'ctmatch-retriever-v2')

# ===== MODEL =================================================================
BASE_MODEL     = 'sentence-transformers/all-MiniLM-L6-v2'          # off-the-shelf baseline
HUB_REPO       = 'semaj83/ctmatch-retriever-v2'

# ===== TRAINING ==============================================================
BATCH_SIZE     = 64      # in-batch negatives: larger batch → harder negatives
EPOCHS         = 3
LR             = 2e-5
WARMUP_RATIO   = 0.1

# ===== EVAL ==================================================================
NDCG_K         = 10
RECALL_KS      = [10, 50, 100]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')
print(f'Train file exists: {os.path.exists(TRAIN_FILE)}')
print(f'Qrels file exists: {os.path.exists(QRELS_FILE)}')

## 1. Training Data

Extract positive (topic, trial) pairs from `train_clf_data.jsonl` (label=2 only).
TREC22 is already excluded from this file — it's the clean train split.
`MultipleNegativesRankingLoss` treats other batch elements as in-batch negatives,
so larger batch sizes → harder negatives. No explicit negative sampling needed.

In [ ]:
import json
from collections import Counter

all_pairs = []
label_counts = Counter()
with open(TRAIN_FILE) as f:
    for line in f:
        ex = json.loads(line)
        label_counts[int(ex['label'])] += 1
        all_pairs.append(ex)

positives = [(ex['topic'], ex['doc']) for ex in all_pairs if int(ex['label']) == 2]

print(f'Total pairs:     {len(all_pairs):,}')
print(f'Label dist:      {dict(sorted(label_counts.items()))}')
print(f'Positive pairs:  {len(positives):,}  (label=2, used for training)')
print(f'\nExample positive pair:')
print(f'  Topic: {positives[0][0][:100]}...')
print(f'  Doc:   {positives[0][1][:100]}...')

## 2. Fine-tuning

Uses `MultipleNegativesRankingLoss` (MNRL): treats all other (topic, doc) pairs in the
batch as negatives. With batch_size=64, each example sees 63 in-batch negatives.

This is the standard contrastive objective for bi-encoder retrieval — equivalent to
the in-batch NCE loss used in DPR, E5, and other dense retrievers.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

print(f'Loading {BASE_MODEL}...')
model = SentenceTransformer(BASE_MODEL)

examples = [InputExample(texts=[topic, doc]) for topic, doc in positives]
dataloader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss = MultipleNegativesRankingLoss(model)

warmup_steps = int(len(dataloader) * EPOCHS * WARMUP_RATIO)
print(f'Training: {len(examples):,} examples, {len(dataloader)} steps/epoch, {EPOCHS} epochs')
print(f'Warmup steps: {warmup_steps}')

model.fit(
    train_objectives=[(dataloader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LR},
    show_progress_bar=True,
    save_best_model=False,
    output_path=OUTPUT_DIR,
)

print(f'\nSaved to {OUTPUT_DIR}')

## 3. Save & Push to Hub

In [ ]:
from huggingface_hub import HfApi

# Reload from saved path to confirm it serialized correctly
finetuned_model = SentenceTransformer(OUTPUT_DIR)
print(f'Reloaded fine-tuned model from {OUTPUT_DIR}')

finetuned_model.save_to_hub(
    repo_id=HUB_REPO,
    token=os.environ.get('HF_TOKEN'),
    exist_ok=True,
)
print(f'Pushed → https://huggingface.co/{HUB_REPO}')

## 4. Judged-Pool Retrieval Evaluation

Measures bi-encoder ranking quality **within the TREC22 judged pool** (35,394 pairs, 50 topics).
For each topic:
- Embed all judged docs with the model
- Rank by cosine similarity to topic embedding
- Compute NDCG@10 and MRR against qrel labels

**Important limitation:** This is a *proxy* metric, not full-corpus retrieval recall.
The judged pool is enriched (all judged docs included regardless of retrieval score),
so the NDCG here is higher than what the bi-encoder would achieve in live inference
where it must retrieve relevant docs from 374k unseen documents. Full-corpus retrieval
evaluation (recall@1000 over 374k) requires re-embedding the corpus and is deferred.

This metric directly answers: *does the fine-tuned model rank relevant docs higher
than irrelevant docs within the same judged pool that the classifiers are evaluated on?*

In [ ]:
import json
from collections import defaultdict

# unified_qrels.jsonl schema: {source, topic_id, topic_text, doc_id, label}
# No doc_text field — texts are loaded separately from semaj83/ctmatch_ir
trec22_by_topic = defaultdict(list)
with open(QRELS_FILE) as f:
    for line in f:
        ex = json.loads(line)
        if ex['topic_id'].startswith('trec22_'):
            trec22_by_topic[ex['topic_id']].append(ex)

n_topics = len(trec22_by_topic)
n_pairs  = sum(len(v) for v in trec22_by_topic.values())
n_rel    = sum(1 for v in trec22_by_topic.values() for ex in v if int(ex['label']) == 2)
print(f'TREC22 judged pool: {n_topics} topics, {n_pairs:,} pairs, {n_rel:,} relevant (label=2)')

sample_topic = next(iter(trec22_by_topic))
print(f'Sample topic: {sample_topic}, {len(trec22_by_topic[sample_topic])} docs')

In [ ]:
# semaj83/ctmatch_ir stores corpus as two parallel .txt files:
#   index2docid.txt  — one NCT ID per line
#   doc_texts.txt    — one doc text per line (same order)
from datasets import load_dataset

print('Loading doc texts from semaj83/ctmatch_ir (~374k docs, ~1 min)...')
ids_ds   = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
texts_ds = load_dataset('semaj83/ctmatch_ir', data_files='doc_texts.txt',  split='train')

docid2text = dict(zip(ids_ds['text'], texts_ds['text']))
print(f'Loaded {len(docid2text):,} doc texts')

# Join doc_text into each qrel record
for examples in trec22_by_topic.values():
    for ex in examples:
        ex['doc_text'] = docid2text.get(ex['doc_id'], '')

n_missing = sum(1 for exs in trec22_by_topic.values() for ex in exs if not ex['doc_text'])
print(f'Missing texts: {n_missing} (expect 0)')

In [ ]:
import numpy as np
from tqdm.auto import tqdm


def ndcg_at_k(ranked_ids, relevances, k=10):
    dcg = sum(
        (2 ** relevances.get(doc_id, 0) - 1) / np.log2(i + 2)
        for i, doc_id in enumerate(ranked_ids[:k])
    )
    ideal = sorted(relevances.values(), reverse=True)[:k]
    idcg = sum((2 ** r - 1) / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def mrr_score(ranked_ids, relevances, relevant_label=2):
    for i, doc_id in enumerate(ranked_ids):
        if relevances.get(doc_id, 0) >= relevant_label:
            return 1.0 / (i + 1)
    return 0.0


def recall_at_k(ranked_ids, relevances, k, relevant_label=2):
    n_relevant = sum(1 for v in relevances.values() if v >= relevant_label)
    if n_relevant == 0:
        return 0.0
    n_retrieved = sum(1 for doc_id in ranked_ids[:k] if relevances.get(doc_id, 0) >= relevant_label)
    return n_retrieved / n_relevant


def eval_biencoder(st_model, topics_dict, batch_size=256):
    """Evaluate SentenceTransformer on TREC22 judged pool."""
    ndcgs, mrrs, recalls = [], [], {k: [] for k in RECALL_KS}

    for topic_id, examples in tqdm(topics_dict.items(), desc='Evaluating topics'):
        topic_text = examples[0]['topic_text']
        doc_ids    = [ex['doc_id']   for ex in examples]
        doc_texts  = [ex['doc_text'] for ex in examples]
        relevances = {ex['doc_id']: int(ex['label']) for ex in examples}

        # Encode topic and all judged docs
        topic_emb = st_model.encode(topic_text, convert_to_numpy=True, normalize_embeddings=True)
        doc_embs  = st_model.encode(doc_texts,  convert_to_numpy=True, normalize_embeddings=True,
                                    batch_size=batch_size, show_progress_bar=False)

        # Cosine similarity (dot product after L2-normalization)
        sims = doc_embs @ topic_emb
        ranked_ids = [doc_ids[i] for i in np.argsort(-sims)]

        ndcgs.append(ndcg_at_k(ranked_ids, relevances, k=NDCG_K))
        mrrs.append(mrr_score(ranked_ids, relevances))
        for k in RECALL_KS:
            recalls[k].append(recall_at_k(ranked_ids, relevances, k=k))

    recall_means = {k: float(np.mean(v)) for k, v in recalls.items()}
    return {
        f'ndcg@{NDCG_K}': float(np.mean(ndcgs)),
        'mrr':             float(np.mean(mrrs)),
        **{f'recall@{k}': recall_means[k] for k in RECALL_KS},
    }

In [ ]:
import pandas as pd

results = {}

# --- Off-the-shelf MiniLM (baseline) ---
print('Evaluating off-the-shelf MiniLM...')
baseline_model = SentenceTransformer(BASE_MODEL)
results['MiniLM (off-the-shelf)'] = eval_biencoder(baseline_model, trec22_by_topic)
print(results['MiniLM (off-the-shelf)'])

# --- Fine-tuned MiniLM ---
print('\nEvaluating fine-tuned MiniLM...')
results['MiniLM (fine-tuned, Tier 1b)'] = eval_biencoder(finetuned_model, trec22_by_topic)
print(results['MiniLM (fine-tuned, Tier 1b)'])

## 5. Results

In [ ]:
df = pd.DataFrame(results).T
df.index.name = 'Model'
print(df.round(4).to_string())
print()

# Delta vs baseline
baseline_ndcg = results['MiniLM (off-the-shelf)'][f'ndcg@{NDCG_K}']
ft_ndcg = results['MiniLM (fine-tuned, Tier 1b)'][f'ndcg@{NDCG_K}']
delta = ft_ndcg - baseline_ndcg
sign = '+' if delta >= 0 else ''
print(f'Fine-tuning delta NDCG@{NDCG_K}: {sign}{delta:.4f}')

In [ ]:
# Save retriever results to Drive alongside clf_results.json
RETRIEVER_RESULTS_FILE = os.path.join(DATA_ROOT, 'retriever_results.json')

retriever_results = {
    'MiniLM_off_the_shelf': {
        'model': BASE_MODEL,
        'fine_tuned': False,
        'eval_scope': 'trec22_judged_pool',
        'note': 'Proxy metric: ranking within judged pool, not full-corpus retrieval recall',
        **results['MiniLM (off-the-shelf)'],
    },
    'MiniLM_fine_tuned_tier1b': {
        'model': BASE_MODEL,
        'hub_model_id': HUB_REPO,
        'fine_tuned': True,
        'train_data': 'train_clf_data.jsonl (TREC21+KZ positives, label=2 only)',
        'loss': 'MultipleNegativesRankingLoss',
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'eval_scope': 'trec22_judged_pool',
        'note': 'Proxy metric: ranking within judged pool, not full-corpus retrieval recall',
        **results['MiniLM (fine-tuned, Tier 1b)'],
    },
}

with open(RETRIEVER_RESULTS_FILE, 'w') as f:
    json.dump(retriever_results, f, indent=2)
print(f'Saved → {RETRIEVER_RESULTS_FILE}')

## 6. §7b Numbers for Deep Dive

Copy these numbers into `docs/deep_dive_outline.md` §7b bi-encoder ablation table.
The MedCPT row comes from `reembed_corpus.ipynb`.

In [ ]:
print('=== §7b Table (judged-pool ranking, TREC22, 50 topics) ===')
print(f'{"Model":<40} {"NDCG@10":>8} {"MRR":>8} {"R@10":>8} {"R@100":>8}')
print('-' * 76)
for name, r in results.items():
    print(f"{name:<40} {r[f'ndcg@{NDCG_K}']:>8.4f} {r['mrr']:>8.4f} "
          f"{r['recall@10']:>8.4f} {r['recall@100']:>8.4f}")
print('  MedCPT (off-the-shelf)                  [from reembed_corpus.ipynb]')
print()
print('NOTE: All numbers are within-judged-pool proxy metrics.')
print('Fine-tuning the bi-encoder does not affect clf NDCG in eval_baseline.ipynb')
print('(eval mode collapses cascade — all judged docs pass through regardless).')
print('Real benefit is higher recall@1000 in live inference (full corpus, deferred).')